In [ ]:
import numpy as np

In [ ]:
dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_clustering"


# comparison in between amp factors, phase shifts in power and phase

In [ ]:
freq_bands = {
              "theta": (0, 4),
              "delta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
amplification_factors = [0.5,0.8,0.9,1.1,1.2,1.5]
phase_peturbations = np.arange(45, 316, 45)

# compute similartiy of similarity matrices with Mantel test

## for pearson correlation



### for non-abs

In [ ]:
import Mantel
import matplotlib.pyplot as plt

In [ ]:
def load_distance_matrices(is_abs = False, correlation_type = "spearman"):
    load_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_clustering/distance_matrices/new_distance_matrices2"
    dist_matrices_phase = {}
    dist_matrices_power = {}
    for freq_band in freq_bands:
        dist_matrices_power[freq_band] = {}
        dist_matrices_phase[freq_band] = {}
        
        if is_abs:
            dist_matrix_power = np.load(f"{load_dir}/power_correlation_matrix_{correlation_type}_{freq_band}_abs.npy", allow_pickle=True).item()
            dist_matrix_phase= np.load(f"{load_dir}/phase_correlation_matrix_{correlation_type}_{freq_band}_abs.npy", allow_pickle=True).item()
        else:
            dist_matrix_power = np.load(f"{load_dir}/power_correlation_matrix_{correlation_type}_{freq_band}.npy", allow_pickle=True).item()
            dist_matrix_phase= np.load(f"{load_dir}/phase_correlation_matrix_{correlation_type}_{freq_band}.npy", allow_pickle=True).item()
            
  
        dist_matrices_power[freq_band] = dist_matrix_power
        dist_matrices_phase[freq_band]= dist_matrix_phase

    return dist_matrices_phase, dist_matrices_power

In [ ]:
dist_matrices_phase, dist_matrices_power = load_distance_matrices()

In [ ]:
dist_matrices_phase["theta"].shape

## within band and perturbation method but for different amplification factors

In [ ]:
import itertools
def compute_Mantel_within_band(dist_matrices, amplification_factors):
    results = {}

    for freq in freq_bands:
        results[freq] = {}
        for factor1, factor2 in itertools.combinations(amplification_factors, 2):
            dist1 = dist_matrices[freq][factor1]
            dist2 = dist_matrices[freq][factor2]
            
            mantel = Mantel.test(dist1, dist2, method='pearson', perms=10000)
            results[freq][(factor1, factor2)] = mantel


    
    return results

In [ ]:
amplification_factors

In [ ]:
results_power = compute_Mantel_within_band(dist_matrices_power, amplification_factors)
results_phase = compute_Mantel_within_band(dist_matrices_phase, phase_peturbations)

In [ ]:
def get_mantel_summary_stats(results):
    avg_results = {}
    std_results = {}
    for freq in freq_bands:
        avg_results[freq] = np.mean([result[0] for result in results[freq].values()])
        std_results[freq] = np.std([result[0] for result in results[freq].values()])

    return avg_results, std_results

In [ ]:
avg_results_power, std_results_power = get_mantel_summary_stats(results_power)
avg_results_phase, std_results_phase =  get_mantel_summary_stats(results_phase)

In [ ]:
np.arange(0,1.1,0.2)

In [ ]:
def plot_summary_stats(avg_results, std_results, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, 4))
    
    bars = ax.bar(avg_results.keys(), avg_results.values(), yerr=std_results.values(), capsize=5)
    ax.set_title(title, fontsize=20)
    ax.set_ylabel('Pearson Correlation', fontsize=16)
    ax.set_xlabel('Frequency Band', fontsize=16)
    ax.set_xticklabels(avg_results.keys(), fontsize=14)
    ax.set_yticklabels(np.round(np.arange(0,1.1,0.2),1), fontsize=14)
    ax.set_ylim(0, 1)
    
    return bars, ax

In [ ]:
print("Power amplification correlation results:")
print("----------------------------------------")
for freq_band in freq_bands:
    print(f"{freq_band}: avg = {avg_results_power[freq_band]:.4f}, std = {std_results_power[freq_band]:.4f}")

print("\nPhase perturbation correlation results:")
print("----------------------------------------")
for freq_band in freq_bands:
    print(f"{freq_band}: avg = {avg_results_phase[freq_band]:.4f}, std = {std_results_phase[freq_band]:.4f}")


In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.3)
results = [(avg_results_power, std_results_power),
           (avg_results_phase, std_results_phase)]
titles = ['power amplifications', 'phase perturbations']
for ax, title, result in zip(axs, titles, results):
    plot_summary_stats(result[0], result[1], f'Average Correlation across {title}', ax=ax)
fig.savefig("distance_matrices/correlations_cross_changes.png")

for power the specific amplification factor does not seem to matter much For the ranking of channels. For the phase shift on the other hand it matters a lot by how much we shif the phase!

## similarity for each pair of frequency and power


In [ ]:
def compute_Mantel_across_methods(dist_matrices_phase, dist_matrices_power, show_first_plot=True, ax=None):

    results_within_freq = {}
    
    for freq in freq_bands:
        results_within_freq[freq] = {}
        for factor in amplification_factors:
            dist_power = dist_matrices_power[freq][factor]
            for phase in phase_peturbations:
                dist_phase = dist_matrices_phase[freq][phase]
                # for each pair of amplifcation factor in power and phase shift in phase, compute the Mantel test on their similarity matrices
                mantel = Mantel.test(dist_power, dist_phase, method='pearson', perms=1000)
                results_within_freq[freq][(factor, phase)] = mantel
    
    avg_within_freq = {}
    std_within_freq = {}
    
    for freq in freq_bands:
        # take the mean and std of the Mantel test results for each frequency band (mean over all pairs of amplification factor and phase shift)
        avg_within_freq[freq] = np.mean([result[0] for result in results_within_freq[freq].values()])
        std_within_freq[freq] = np.std([result[0] for result in results_within_freq[freq].values()])
    

    power_cross_freq = {}
    phase_cross_freq = {}

    for i, freq1 in enumerate(freq_bands):
        for j, freq2 in enumerate(freq_bands):
            if j > i: 
                key = f"{freq1} vs {freq2}"
                power_cross_freq[key] = []
                phase_cross_freq[key] = []
                
                # Compare power matrices across frequencies
                for factor in amplification_factors:
                    dist_power1 = dist_matrices_power[freq1][factor]
                    dist_power2 = dist_matrices_power[freq2][factor]
                    mantel = Mantel.test(dist_power1, dist_power2, method='pearson', perms=1000)
                    power_cross_freq[key].append(mantel[0])
                
                # Compare phase matrices across frequencies
                for phase in phase_peturbations:
                    dist_phase1 = dist_matrices_phase[freq1][phase]
                    dist_phase2 = dist_matrices_phase[freq2][phase]
                    mantel = Mantel.test(dist_phase1, dist_phase2, method='pearson', perms=1000)
                    phase_cross_freq[key].append(mantel[0])
    

    avg_power_cross = {key: np.mean(values) for key, values in power_cross_freq.items()}
    std_power_cross = {key: np.std(values) for key, values in power_cross_freq.items()}
    avg_phase_cross = {key: np.mean(values) for key, values in phase_cross_freq.items()}
    std_phase_cross = {key: np.std(values) for key, values in phase_cross_freq.items()}
    
    if show_first_plot:
        fig, axes = plt.subplots(2, 1, figsize=(16, 8))
        
        # Plot within-frequency results
        plot_summary_stats(avg_within_freq, std_within_freq, 'Average power vs phase correlation', ax=axes[0])
        
        # Set up second plot in the existing figure
        ax_cross_freq = axes[1]
    else:
        if ax is None:
            fig, ax_cross_freq = plt.subplots(figsize=(16, 5))
        else:
            ax_cross_freq = ax
            fig = ax_cross_freq.figure

    x = np.arange(len(avg_power_cross))
    width = 0.45

    ax_cross_freq.bar(x - width/2, list(avg_power_cross.values()), width, yerr=list(std_power_cross.values()), 
                    capsize=5, label='Power', color='#FF9500')  # Bright orange
    ax_cross_freq.bar(x + width/2, list(avg_phase_cross.values()), width, yerr=list(std_phase_cross.values()), 
                    capsize=5, label='Phase', color='#00C3E3')  # Teal blue
    
    ax_cross_freq.set_title('Average pairwise correlation between frequency bands', fontsize=22)
    ax_cross_freq.set_ylabel('Correlation', fontsize=20)
    ax_cross_freq.set_xlabel('Frequency Band Pairs', fontsize=20)
    ax_cross_freq.set_xticks(x)
    ax_cross_freq.set_xticklabels(list(avg_power_cross.keys()), fontsize=18, rotation=45, ha='right')
   
    ax_cross_freq.tick_params(axis='y', labelsize=18)
    ax_cross_freq.legend(fontsize=18)
    
    plt.tight_layout()
    if show_first_plot:
        plt.subplots_adjust(hspace=0.4)
    
    # Only save figure if we created it within this function
    if not (not show_first_plot and ax is not None):
        fig.savefig("distance_matrices/correlations_within_cross_freq.png")
    
    return {
        'within_freq': results_within_freq,
        'avg_within_freq': avg_within_freq,
        'std_within_freq': std_within_freq,
        'avg_power_cross': avg_power_cross,
        'std_power_cross': std_power_cross,
        'avg_phase_cross': avg_phase_cross,
        'std_phase_cross': std_phase_cross
    }

In [ ]:
result = compute_Mantel_across_methods(dist_matrices_phase, dist_matrices_power, show_first_plot=False)

In [ ]:
result.keys()

In [ ]:
result["avg_within_freq"]

In [ ]:
result["std_within_freq"]

In [ ]:
result = compute_Mantel_across_methods(dist_matrices_phase, dist_matrices_power, show_first_plot=False)

For most frequency bands, power and phase values are correlated highly, though much less compared to the absolute values

### for abs

In [ ]:
dist_matrices_phase, dist_matrices_power = load_distance_matrices(is_abs=True)

In [ ]:
results_power = compute_Mantel_within_band(dist_matrices_power, amplification_factors)
results_phase = compute_Mantel_within_band(dist_matrices_phase, phase_peturbations)

In [ ]:
avg_results_power, std_results_power = get_mantel_summary_stats(results_power)
avg_results_phase, std_results_phase =  get_mantel_summary_stats(results_phase)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.3)
results = [(avg_results_power, std_results_power),
           (avg_results_phase, std_results_phase)]
titles = ['power amplifications', 'phase perturbations']
for ax, title, result in zip(axs, titles, results):
    plot_summary_stats(result[0], result[1], f'Average Correlation across {title}', ax=ax)
fig.savefig("distance_matrices/correlations_cross_changes.png")

In [ ]:
print("Power amplification correlation results:")
print("----------------------------------------")
for freq_band in freq_bands:
    print(f"{freq_band}: avg = {avg_results_power[freq_band]:.4f}, std = {std_results_power[freq_band]:.4f}")

print("\nPhase perturbation correlation results:")
print("----------------------------------------")
for freq_band in freq_bands:
    print(f"{freq_band}: avg = {avg_results_phase[freq_band]:.4f}, std = {std_results_phase[freq_band]:.4f}")


In [ ]:
result = compute_Mantel_across_methods(dist_matrices_phase, dist_matrices_power, show_first_plot=False)

## repeat test with spearman correlation

## non abs

In [ ]:
dist_matrices_phase, dist_matrices_power = load_distance_matrices(correlation_type='spearman')
results_power = compute_Mantel_within_band(dist_matrices_power, amplification_factors)
results_phase = compute_Mantel_within_band(dist_matrices_phase, phase_peturbations)

In [ ]:
avg_results_power, std_results_power = get_mantel_summary_stats(results_power)
avg_results_phase, std_results_phase =  get_mantel_summary_stats(results_phase)

In [ ]:
print("Power amplification correlation results:")
print("----------------------------------------")
for freq_band in freq_bands:
    print(f"{freq_band}: avg = {avg_results_power[freq_band]:.4f}, std = {std_results_power[freq_band]:.4f}")

print("\nPhase perturbation correlation results:")
print("----------------------------------------")
for freq_band in freq_bands:
    print(f"{freq_band}: avg = {avg_results_phase[freq_band]:.4f}, std = {std_results_phase[freq_band]:.4f}")


In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.3)
results = [(avg_results_power, std_results_power),
           (avg_results_phase, std_results_phase)]
titles = ['power amplifications', 'phase perturbations']
for ax, title, result in zip(axs, titles, results):
    plot_summary_stats(result[0], result[1], f'Average Correlation across {title}', ax=ax)
fig.savefig("distance_matrices/correlations_cross_changes.png")

In [ ]:
result = compute_Mantel_across_methods(dist_matrices_phase, dist_matrices_power, show_first_plot=True)

In [ ]:
result.keys()

In [ ]:
result["avg_within_freq"], result["std_within_freq"]

### abs

In [ ]:
dist_matrices_phase_abs, dist_matrices_power_abs = load_distance_matrices(correlation_type='spearman', is_abs=True)
results_power_abs = compute_Mantel_within_band(dist_matrices_power_abs, amplification_factors)
results_phase_abs = compute_Mantel_within_band(dist_matrices_phase_abs, phase_peturbations)

In [ ]:
avg_results_power_abs, std_results_power_abs = get_mantel_summary_stats(results_power_abs)
avg_results_phase_abs, std_results_phase_abs =  get_mantel_summary_stats(results_phase_abs)

In [ ]:
print("Power amplification correlation results:")
print("----------------------------------------")
for freq_band in freq_bands:
    print(f"{freq_band}: avg = {avg_results_power_abs[freq_band]:.4f}, std = {std_results_power_abs[freq_band]:.4f}")

print("\nPhase perturbation correlation results:")
print("----------------------------------------")
for freq_band in freq_bands:
    print(f"{freq_band}: avg = {avg_results_phase_abs[freq_band]:.4f}, std = {std_results_phase_abs[freq_band]:.4f}")

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, sharey=True)
plt.subplots_adjust(hspace=0.3)
results = [(avg_results_power_abs, std_results_power_abs),
           (avg_results_phase_abs, std_results_phase_abs)]
titles = ['power amplifications', 'phase perturbations']
for ax, title, result in zip(axs, titles, results):
    plot_summary_stats(result[0], result[1], f'Average Correlation across {title}', ax=ax)
fig.savefig("distance_matrices/correlations_cross_changes.png")

In [ ]:

result_abs = compute_Mantel_across_methods(dist_matrices_phase_abs, dist_matrices_power_abs, show_first_plot=True)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(16, 8), sharex=True, sharey=True)
compute_Mantel_across_methods(dist_matrices_phase, dist_matrices_power, show_first_plot=False, ax=axs[0])


axs[0].set_title('')
axs[0].set_xlabel('')

compute_Mantel_across_methods(dist_matrices_phase_abs, dist_matrices_power_abs, show_first_plot=False, ax=axs[1])
axs[1].set_title('')
axs[1].legend().remove()
fig.suptitle('Correlation of Similarity Matrices for band pairs (raw vs abs)', fontsize=24)

fig.savefig("distance_matrices/correlations_cross_freq.png")

# what is going on for phase in non abs part? 

In [ ]:
dist_matrices_phase, _ = load_distance_matrices()

In [ ]:
results_phase = compute_Mantel_within_band(dist_matrices_phase, phase_peturbations)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

def create_phase_correlation_matrices():
    # Initialize matrices for each frequency band
    correlation_matrices = {}
    
    # Extract correlation values from the results_phase dictionary
    for freq in freq_bands:
        # Create empty matrix for this frequency band
        n = len(phase_peturbations)
        corr_matrix = np.zeros((n, n))
        
        # Fill in the matrix with correlation values
        for i, phase1 in enumerate(phase_peturbations):
            for j, phase2 in enumerate(phase_peturbations):
                if i == j:
                    # Diagonal elements (correlation with self) are 1
                    corr_matrix[i, j] = 1.0
                elif i < j:
                    # Get the correlation from results_phase
                    if (phase1, phase2) in results_phase[freq]:
                        corr_matrix[i, j] = results_phase[freq][(phase1, phase2)][0]
                    else:
                        # If reverse order pair exists
                        corr_matrix[i, j] = results_phase[freq][(phase2, phase1)][0]
                else:
                    # Mirror the upper triangular part
                    corr_matrix[i, j] = corr_matrix[j, i]
        
        correlation_matrices[freq] = corr_matrix
    
    # Plot heatmaps for each frequency band
    fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
    
    # Create a mask for 180-degree shifts
    mask_180 = np.zeros((len(phase_peturbations), len(phase_peturbations)))
    for i, p1 in enumerate(phase_peturbations):
        for j, p2 in enumerate(phase_peturbations):
            # Check if phases are 180 degrees apart (or closest approximation)
            if abs((p1 + 180) % 360 - p2) < 1 or abs((p2 + 180) % 360 - p1) < 1:
                mask_180[i, j] = 1
    
    # Custom annotation function to color text in 180-degree shift cells
    def custom_text_color(val, i, j):
        if mask_180[i, j] == 1:
            return 'red'
        return 'black'
    
    for i, (freq, matrix) in enumerate(correlation_matrices.items()):
        ax = axes[i]
        sns_heatmap = sns.heatmap(matrix, annot=True, fmt=".3f", cmap="YlGnBu", 
                   xticklabels=phase_peturbations, yticklabels=phase_peturbations,
                   vmin=0, vmax=1, ax=ax, cbar=(i==4))
        
        # Update text color for cells that are 180 degrees apart
        for text, (i_txt, j_txt) in zip(ax.texts, [(i, j) for i in range(len(phase_peturbations)) 
                                                 for j in range(len(phase_peturbations))]):
            if mask_180[i_txt, j_txt] == 1:
                text.set_color('red')
                text.set_weight('bold')
        
        ax.set_title(f"{freq}")
        ax.set_xlabel("Phase Perturbation (°)")
        if i == 0:
            ax.set_ylabel("Phase Perturbation (°)")
    
    plt.suptitle("Correlation between Phase Perturbation Distance Matrices", fontsize=16)
    plt.tight_layout()
    plt.savefig("distance_matrices/phase_perturbation_correlations.png", dpi=300)
    plt.show()
    
    return correlation_matrices

phase_correlation_matrices = create_phase_correlation_matrices()

The second off diagonal which represent 180 degree shifts is often the one with the lowest correlation, showing a sort of cyclical effects in the theta, delta (and alpha) bands.

In [ ]:
dist_matrices_phase, _ = load_distance_matrices(correlation_type='spearman')
results_phase = compute_Mantel_within_band(dist_matrices_phase, phase_peturbations)

In [ ]:
phase_correlation_matrices = create_phase_correlation_matrices()

# compare rank correlation similarity

## gradshap

In [ ]:
import numpy as np
import Mantel

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
freq_bands = {
              "theta": (0, 4),
              "delta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
amplification_factors = [0.5, 0.8, 0.9, 1.1, 1.2, 1.5]
phase_peturbations = np.arange(45, 316, 45)

In [ ]:

def load_rank_correlation_distance_matrices(exp_method="gradshap", correlation_type="spearman", take_abs=False):
    load_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_clustering/distance_matrices/new_distance_matrices2"
    features = ["phase","power"]

    features_distance_matrics = {}
    top_k_distance_matrices = {}
    for freq_band in freq_bands:
        features_distance_matrics[freq_band] = {}
        for feature in features:
            features_distance_matrics[freq_band][feature] = {}
            if take_abs:
                if feature == "power":
                    features_distance_matrics[freq_band][feature] = np.load(f"{load_dir}/power_correlation_matrix_{correlation_type}_{freq_band}_abs.npy", allow_pickle=True).item()
                elif feature == "phase":
                    features_distance_matrics[freq_band][feature] = np.load(f"{load_dir}/phase_correlation_matrix_{correlation_type}_{freq_band}_abs.npy", allow_pickle=True).item()
            else:
                if feature == "power":
                    features_distance_matrics[freq_band][feature] = np.load(f"{load_dir}/power_correlation_matrix_{correlation_type}_{freq_band}.npy", allow_pickle=True).item()
                else:
                    features_distance_matrics[freq_band][feature] = np.load(f"{load_dir}/phase_correlation_matrix_{correlation_type}_{freq_band}.npy", allow_pickle=True).item()

    load_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_clustering/distance_matrices/new_distance_matrices2"
    top_k_distance_matrix = {}
    if take_abs:
        top_k_distance_matrix = np.load(f"{load_dir}/top_k_new_{exp_method}_abs_{correlation_type}.npy")
    else:
        top_k_distance_matrix = np.load(f"{load_dir}/top_k_new_{exp_method}_{correlation_type}.npy")
    
    return features_distance_matrics, top_k_distance_matrix
       

In [ ]:
amplification_factors

In [ ]:
def compute_mantel_feature_vs_topk(features_distance_matrics, top_k_distance_matrix):

    features = ["phase", "power"]

    all_results = {}
  
    for feature in features:
        all_results[feature] = {}
        for freq_band in freq_bands:
            all_results[feature][freq_band] = {}
            if feature == "phase":
                for phase in phase_peturbations:
                    all_results[feature][freq_band][phase] = Mantel.test(features_distance_matrics[freq_band][feature][phase], top_k_distance_matrix, method='pearson', perms=10000)[0]

            elif feature=="power":
                for factor in amplification_factors:
                    all_results[feature][freq_band][factor] = Mantel.test(features_distance_matrics[freq_band][feature][factor], top_k_distance_matrix, method='pearson', perms=10000)[0]
    return all_results
  

   

In [ ]:
def plot_mantel_correlation_by_freq_band(results_dict, frequency_band=None):
    """
    Plot Mantel correlation for power and phase in separate subplots across frequency bands.
    
    Parameters:
    results_dict (dict): Dictionary with results from compute_mantel_feature_vs_topk
    frequency_band (str, optional): If provided, plot only this specific frequency band.
    """
    # Determine which frequency bands to plot
    bands_to_plot = [frequency_band] if frequency_band is not None else freq_bands.keys()
    num_bands = len(bands_to_plot)
    
    # Create a figure with 2 columns (power and phase) and rows for each frequency band
    fig, axes = plt.subplots(num_bands, 2, figsize=(14, 4 * num_bands), sharey=True)
    
    # Handle single band case
    if num_bands == 1:
        axes = axes.reshape(1, 2)
    
    # Get colors from seaborn palette
    colors = sns.color_palette("Set2", 2)
    
    for i, freq_band in enumerate(bands_to_plot):
        # Plot power correlation
        power_values = list(results_dict['power'][freq_band].values())
        power_labels = [str(factor) for factor in amplification_factors]
        
        axes[i, 0].bar(range(len(power_values)), power_values, color=colors[0])
        axes[i, 0].set_xticks(range(len(power_values)))
        axes[i, 0].set_xticklabels(power_labels, rotation=45, ha='right', fontsize=15)
        axes[i, 0].set_ylabel('Mantel Correlation', fontsize=17)
        axes[i, 0].tick_params(axis='y', labelsize=15)
        axes[i, 0].set_ylim(0, 0.8)
        axes[i, 0].set_title(f'Power', fontsize=16)
        
        # Plot phase correlation
        phase_values = list(results_dict['phase'][freq_band].values())
        phase_labels = [str(phase) + '°' for phase in phase_peturbations]
        
        axes[i, 1].bar(range(len(phase_values)), phase_values, color=colors[1])
        axes[i, 1].set_xticks(range(len(phase_values)))
        axes[i, 1].set_xticklabels(phase_labels, rotation=45, ha='right', fontsize=15)
        #axes[i, 1].set_ylabel('Mantel Correlation', fontsize=17)
        axes[i, 1].set_ylim(0, 0.8)
        axes[i, 1].set_title(f'Phase', fontsize=16)
    
    plt.tight_layout()
    
    # Add overall title
    if frequency_band is None:
        fig.suptitle('Mantel Correlation between Feature Similarity Matrices and Top-K Similarity Matrix', 
                  fontsize=18, y=1.02)
        save_path = "distance_matrices/mantel_correlation_by_freq_band_separate.png"
    else:
        fig.suptitle(f'Mantel Correlation {frequency_band.capitalize()} Band (feature vs explanation)', 
                  fontsize=18, y=1.04)
        save_path = f"distance_matrices/mantel_correlation_{frequency_band}_band_separate.png"
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return fig, axes


In [ ]:
features_distance_matrics, top_k_distance_matrix = load_rank_correlation_distance_matrices()
results = compute_mantel_feature_vs_topk(features_distance_matrics, top_k_distance_matrix)
features_distance_matrics_abs, top_k_distance_matrix_abs = load_rank_correlation_distance_matrices(take_abs=True)
results_abs = compute_mantel_feature_vs_topk(features_distance_matrics_abs, top_k_distance_matrix_abs)

### non abs

In [ ]:
fig, axes = plot_mantel_correlation_by_freq_band(results, frequency_band="gamma")

### abs

In [ ]:
fig, axes = plot_mantel_correlation_by_freq_band(results_abs)

In [ ]:
results_abs

In [ ]:
def plot_mantel_correlation_averaged(results_abs):
    # average results across phase perturbations and amplification factors
    # then a single plots is shown using the averaged results

    # Initialize dictionaries to store average results
    avg_power = {}
    avg_phase = {}
    std_power = {}
    std_phase = {}

    # Calculate average and std for each frequency band
    for freq_band in freq_bands:
        # Average across amplification factors for power
        power_values = list(results_abs['power'][freq_band].values())
        avg_power[freq_band] = np.mean(power_values)
        std_power[freq_band] = np.std(power_values)
        
        # Average across phase perturbations for phase
        phase_values = list(results_abs['phase'][freq_band].values())
        avg_phase[freq_band] = np.mean(phase_values)
        std_phase[freq_band] = np.std(phase_values)

    # Create figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    # Get colors from seaborn palette
    colors = sns.color_palette("Set2", 2)

    # Plot power results
    axes[0].bar(range(len(avg_power)), list(avg_power.values()), yerr=list(std_power.values()), 
                color=colors[0], capsize=5)
    axes[0].set_xticks(range(len(avg_power)))
    axes[0].set_xticklabels(list(avg_power.keys()), rotation=45, ha='right')
    axes[0].set_ylabel('Average Mantel Correlation', fontsize=16)
    axes[0].set_title('Power', fontsize=18)
    axes[0].set_ylim(0, 0.8)  # Consistent with previous plots
    axes[0].tick_params(axis='y', labelsize=16)
    axes[0].tick_params(axis='x', labelsize=16)

    # Plot phase results
    axes[1].bar(range(len(avg_phase)), list(avg_phase.values()), yerr=list(std_phase.values()), 
                color=colors[1], capsize=5)
    axes[1].set_xticks(range(len(avg_phase)))
    axes[1].set_xticklabels(list(avg_phase.keys()), rotation=45, ha='right')
    axes[1].set_title('Phase', fontsize=18)
    axes[1].tick_params(axis='y', labelsize=16)
    axes[1].tick_params(axis='x', labelsize=16)
    # Add overall title
    fig.suptitle('Average Mantel Correlation (abs feature vs abs explanaiton function)', fontsize=20)

    plt.tight_layout()
    plt.savefig("distance_matrices/average_mantel_correlation.png", dpi=300, bbox_inches='tight')

    return fig, axes

In [ ]:
plot_mantel_correlation_averaged(results_abs)

## saliency

### non abs

In [ ]:
features_distance_matrics, top_k_distance_matrix = load_rank_correlation_distance_matrices(exp_method="saliency")
results = compute_mantel_feature_vs_topk(features_distance_matrics, top_k_distance_matrix)
fig, axes = plot_mantel_correlation_by_freq_band(results)


### abs

In [ ]:
features_distance_matrics_abs, top_k_distance_matrix_abs = load_rank_correlation_distance_matrices(exp_method="saliency", take_abs=True)
results_abs = compute_mantel_feature_vs_topk(features_distance_matrics_abs, top_k_distance_matrix_abs)
fig, axes = plot_mantel_correlation_by_freq_band(results_abs)

In [ ]:
def compute_mantel_feature_vs_topk(correlations=["pearson", "spearman"]):

    features = ["phase", "power"]
    top_k_types = ["abs", "positive"]
    top_k_methods = ["normalized"]  # Only keep normalized method
    all_results = {}
    
    fig, axes = plt.subplots(1, len(features), figsize=(16, 5), sharey=True)
    
    # Create result matrices for both features and both correlation types
    result_matrices = {}
    for feature in features:
        result_matrix = np.zeros((len(freq_bands), len(correlations) * len(top_k_types)))
        
        for f_i, freq in enumerate(freq_bands):
            col_idx = 0
            for corr in correlations:
                for tk_type in top_k_types:
                    abs_key = "_abs" if tk_type == "abs" else ""
                    feature_dist = features_distance_matrices[freq][feature][corr][abs_key]
                    topk_dist = top_k_distance_matrices[corr][tk_type]["normalized"]
                    mantel_result = Mantel.test(feature_dist, topk_dist, method='pearson', perms=1000)
                    result_matrix[f_i, col_idx] = mantel_result[0]
                    col_idx += 1
        
        all_results[feature] = result_matrix
        result_matrices[feature] = result_matrix

    # Find global min and max for consistent color scaling
    vmin = 0
    vmax = 1
    
    # Create a figure-wide colorbar
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap="YlGnBu", norm=norm)
    sm.set_array([])
    
    # Plot heatmaps
    for f_idx, feature in enumerate(features):
        ax = axes[f_idx]
        
        # Create column labels that show both correlation type and top-k type
        col_labels = []
        for corr in correlations:
            for tk_type in top_k_types:
                if tk_type == "abs":
                    col_labels.append(f"{corr}\nabsolute value")
                else:
                    col_labels.append(f"{corr}")
        
        sns.heatmap(result_matrices[feature], annot=True, fmt=".2f", cmap="YlGnBu", 
                  xticklabels=col_labels, yticklabels=list(freq_bands.keys()),
                  vmin=vmin, vmax=vmax, ax=ax, cbar=False, annot_kws={"fontsize": 16})
        
        ax.set_title(f"{feature.capitalize()}", fontsize=20)
        if f_idx == 0:
            ax.set_ylabel("Frequency Band", fontsize=18)
        ax.tick_params(axis='both', labelsize=16)
        
        # Add a separator line between correlation types
        ax.axvline(x=len(top_k_types), color='black', linestyle='-', linewidth=1)
        
    # Add colorbar with the same height as the plots
    cbar_ax = fig.add_axes([0.92, 0.1, 0.02, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('Correlation', fontsize=16)
    
    plt.suptitle("Mantel Correlation of Similarity Matrices: Features vs Explanation Function", fontsize=20)
    plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # Make room for colorbar and title
    plt.savefig("distance_matrices/feature_vs_topk_mantel_both_correlations.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    return all_results


In [ ]:
compute_mantel_feature_vs_topk(correlations=["pearson", "spearman"])

In [ ]:
compute_mantel_feature_vs_topk(correlations=["spearman"])

### abs